In [79]:
import torch
import os
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [80]:
df = pd.read_csv('parkinsons.data')

In [81]:
with open('important_features.json', 'r') as f:
    important_features = json.load(f)

y = df['status']
X = df[important_features]

In [82]:
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X)
x_test = scaler.transform(X)        #in case test only transform, fit will cause leakage


In [83]:
X_test = X_test.to_numpy()  #since only transform returns df, converting to np.array manually

In [84]:
with open('scaler.pkl', 'wb') as f:
    scaler = pickle.dump(scaler, f)

In [85]:
X_tr_tensor = torch.from_numpy(X_train)
X_ts_tensor = torch.from_numpy(X_test)
y_tr_tensor = y_train.to_numpy(copy=True)
y_ts_tensor = y_test.to_numpy(copy=True)
y_tr_tensor = torch.from_numpy(y_tr_tensor)
y_ts_tensor = torch.from_numpy(y_ts_tensor)


In [86]:
X_ts_tensor.dtype

torch.float64

In [93]:
input_shape = X_tr_tensor.shape[1]
class SimpleNN():
    def __init__(self, X):
        self.weights1 = torch.rand(X.shape[1], 6, requires_grad = True, dtype = torch.float64)
        self.bias1 = torch.zeros(6, requires_grad = True, dtype = torch.float64)
        self.weights2 = torch.rand(6, 4, requires_grad = True, dtype = torch.float64)
        self.bias2 = torch.zeros(4, requires_grad = True, dtype = torch.float64)
        self.weights3 = torch.rand(4, 1, requires_grad = True, dtype = torch.float64)
        self.bias3 = torch.zeros(1, requires_grad = True, dtype = torch.float64)

    def forward(self, X):
        z1 = torch.matmul(X, self.weights1) + self.bias1
        a1 = torch.relu(z1)
        
        z2 = torch.matmul(a1, self.weights2) + self.bias2
        a2 = torch.relu(z2)

        z3 = torch.matmul(a2, self.weights3) + self.bias3
        y_pred = torch.sigmoid(z3)

        return y_pred

    def loss_func(self, y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        # Corrected binary cross-entropy loss
        loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
        return loss

In [94]:
model = SimpleNN(X_tr_tensor)
lr = 0.48
EPOCHS = 25

In [95]:
for i in range(EPOCHS):
    y_pred = model.forward(X_tr_tensor)
    loss = model.loss_func(y_pred, y_tr_tensor)
    print(f"Epoch:{i+1} Loss: {loss.item():.2f}")

    loss.backward()

    with torch.no_grad():
        model.weights1 -= lr*model.weights1.grad
        model.bias1 -= lr*model.bias1.grad
        model.weights2 -= lr*model.weights2.grad
        model.bias2 -= lr*model.bias2.grad
        model.weights3 -= lr*model.weights3.grad
        model.bias3 -= lr*model.bias3.grad
    
    model.weights1.grad.zero_()
    model.bias1.grad.zero_()
    model.weights2.grad.zero_()
    model.bias2.grad.zero_()
    model.weights3.grad.zero_()
    model.bias3.grad.zero_()


Epoch:1 Loss: 1.04
Epoch:2 Loss: 0.81
Epoch:3 Loss: 0.72
Epoch:4 Loss: 1.04
Epoch:5 Loss: 0.85
Epoch:6 Loss: 0.63
Epoch:7 Loss: 0.68
Epoch:8 Loss: 0.90
Epoch:9 Loss: 0.71
Epoch:10 Loss: 0.59
Epoch:11 Loss: 0.59
Epoch:12 Loss: 0.59
Epoch:13 Loss: 0.65
Epoch:14 Loss: 0.58
Epoch:15 Loss: 0.63
Epoch:16 Loss: 0.58
Epoch:17 Loss: 0.63
Epoch:18 Loss: 0.57
Epoch:19 Loss: 0.59
Epoch:20 Loss: 0.58
Epoch:21 Loss: 0.63
Epoch:22 Loss: 0.57
Epoch:23 Loss: 0.57
Epoch:24 Loss: 0.58
Epoch:25 Loss: 0.60


In [96]:
model = SimpleNN(X_ts_tensor)
with torch.no_grad():
    y_pred = model.forward(X_ts_tensor)
    loss = model.loss_func(y_pred, y_ts_tensor)
    print(f"Loss: {loss.item():.2f}")
    

Loss: 3.95


### now time to save the model parameters


In [97]:
torch.save({
    'model_state': {
        'W1': model.weights1.detach(),
        'b1': model.bias1.detach(),
        'W2': model.weights2.detach(),
        'b2': model.bias2.detach(),
        'W3': model.weights3.detach(),
        'b3': model.bias3.detach()
    },
    'input_size': X.shape[1]
}, 'model.pth')